In [3]:
import os
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input, Embedding, LSTM, Dense, Dropout,
    MultiHeadAttention, LayerNormalization, Flatten
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Data Loading & Preprocessing
data_path = 'ArticlesMarch2018.csv'
df = pd.read_csv(data_path)
df['snippet'] = df['snippet'].astype(str)

formatted_text = "\n".join(df['snippet'].tolist())
translator = str.maketrans('', '', string.punctuation)
formatted_text = formatted_text.translate(translator).lower()

# 2. Tokenization & Sequence Creation
tokenizer = Tokenizer()
tokenizer.fit_on_texts([formatted_text])
voc = len(tokenizer.word_index) + 1

input_sequences = []
for line in formatted_text.split('\n'):
    seq = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(seq)):
        input_sequences.append(seq[:i+1])

max_len = max(len(s) for s in input_sequences)
padded = pad_sequences(input_sequences, maxlen=max_len, padding='pre')
X = padded[:, :-1]
y = to_categorical(padded[:, -1], num_classes=voc)

# 3. Callbacks
rlrong = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-5, verbose=1)
estop   = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)

# 4. Build Transformer Teacher
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = Sequential([Dense(ff_dim, activation="relu"), Dense(embed_dim)])
        self.ln1 = LayerNormalization(epsilon=1e-6)
        self.ln2 = LayerNormalization(epsilon=1e-6)
        self.do1 = Dropout(rate)
        self.do2 = Dropout(rate)
    def call(self, inputs, training=None):
        attn = self.att(inputs, inputs)
        attn = self.do1(attn, training=training)
        out1 = self.ln1(inputs + attn)
        ffn = self.ffn(out1)
        ffn = self.do2(ffn, training=training)
        return self.ln2(out1 + ffn)

inp = Input(shape=(max_len-1,))
x = Embedding(voc, 200)(inp)
x = TransformerBlock(200, 4, 128)(x, training=True)
x = LayerNormalization(epsilon=1e-6)(x)
x = Dropout(0.2)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.2)(x)
x = Flatten()(x)
teacher_out = Dense(voc, activation='softmax')(x)
teacher_model = Model(inp, teacher_out, name="teacher")
teacher_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
teacher_model.summary()

# 5. Train Teacher
teacher_history = teacher_model.fit(X, y, epochs=50, validation_split=0.2,
                                    callbacks=[rlrong, estop])

# plt.plot(teacher_history.history['accuracy'], label='Train Accuracy')
# plt.plot(teacher_history.history['val_accuracy'], label='Val Accuracy')
# plt.title('Transformer Teacher Accuracy (Treebank)')
# plt.xlabel('Epoch')
# plt.ylabel('Accuracy')
# plt.legend()
# plt.show()

# =========================
#  Next Word Prediction
# =========================
def predict_next_word(seed_text, next_words=1):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')

        predicted = teacher_model.predict(token_list, verbose=0)
        predicted_word_index = np.argmax(predicted)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_word_index:
                output_word = word
                break

        seed_text += " " + output_word

    return seed_text


print("\nExample Prediction:")
print(predict_next_word("house is", 3))


Model: "teacher"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 40, 200)        │     1,402,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 40, 200)        │       694,928 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_2           │ (None, 40, 200)        │           400 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 40, 200)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 40, 512)        │       102,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 40, 512)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 20480)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7011)           │   143,592,291 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 145,792,731 (556.16 MB)

 Trainable params: 145,792,731 (556.16 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
662/662 ━━━━━━━━━━━━━━━━━━━━ 41s 44ms/step - accuracy: 0.0455 - loss: 7.6439 - val_accuracy: 0.0502 - val_loss: 8.7326 - learning_rate: 0.0010
Epoch 2/50
662/662 ━━━━━━━━━━━━━━━━━━━━ 25s 38ms/step - accuracy: 0.0522 - loss: 7.3387 - val_accuracy: 0.0502 - val_loss: 8.7378 - learning_rate: 0.0010
Epoch 3/50
661/662 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.0513 - loss: 7.3917
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
662/662 ━━━━━━━━━━━━━━━━━━━━ 25s 38ms/step - accuracy: 0.0513 - loss: 7.3917 - val_accuracy: 0.0502 - val_loss: 8.7512 - learning_rate: 0.0010
Epoch 4/50
662/662 ━━━━━━━━━━━━━━━━━━━━ 25s 38ms/step - accuracy: 0.0535 - loss: 7.2595 - val_accuracy: 0.0502 - val_loss: 8.7580 - learning_rate: 2.0000e-04
Epoch 5/50
661/662 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.0543 - loss: 7.2376
Epoch 5: ReduceLROnPlateau reducing learning rate to 4.0000001899898055e-05.
662/662 ━━━━━━━━━━━━━━━━━━━━ 25s 38ms/step - accuracy: 0.0543 -